# DQN and Dyna-DQN: 2,250 m / 120 s, 1,000 episodes

Train both neural agents from fresh weights on the **original random traffic**,
before changing to the controlled overtaking scenario. Each agent chooses driving
and sensing jointly. Route, deadline, rewards, seed, and 12 held-out trials match
`sensor_time_completion_120s_1000episodes.ipynb`. The completed adaptive Expected
SARSA policy is loaded as a matched comparison; it is not retrained.

Run All trains sequentially. A run prepared by `experiments.run_long_dqn` can be
loaded using `FINAL_PROJECT_DQN_RUN_DIR` to avoid duplicate training. Policies are
evaluation checkpoints, not exact training-resume checkpoints.


In [ ]:
from pathlib import Path
import os, sys, json
from dataclasses import asdict
from datetime import datetime
from zoneinfo import ZoneInfo
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "environments/sensor_highway/env.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from environments.sensor_highway import ExperimentConfig
from algorithms.dqn_agent import DQNSettings, DQNAgent
from algorithms.sensor_agent import LinearExpectedSarsa
from experiments.dqn_helpers import train_dqn
from experiments.agent_helpers import evaluate_agents
torch.set_num_threads(1)


In [ ]:
TRAIN_SEED = 42
TRAIN_EPISODES = 1000
TEST_SEEDS = list(range(100_000, 100_004))
TEST_WEATHER = ["clear", "rain", "fog"]
BASE_RUN = PROJECT_ROOT / "runs/training/time_completion_120s_1000ep_20261121_204241_241658"
config = ExperimentConfig.load(BASE_RUN / "config.json")
assert config.highway["duration"] == 120
assert config.reward.route_distance_m == 2250
assert config.highway.get("slow_lead_scenario") is None
settings = DQNSettings(episodes=TRAIN_EPISODES)
assert not set(TEST_SEEDS).intersection(range(TRAIN_SEED, TRAIN_SEED + TRAIN_EPISODES))
existing_run = os.environ.get("FINAL_PROJECT_DQN_RUN_DIR")
if existing_run:
    output = Path(existing_run).resolve()
    saved = json.loads((output / "run_settings.json").read_text())
    assert saved["dqn_settings"] == asdict(settings)
    assert saved["training_seed"] == TRAIN_SEED
    assert saved["test_seeds"] == TEST_SEEDS and saved["test_weather"] == TEST_WEATHER
    assert asdict(ExperimentConfig.load(output / "config.json")) == asdict(config)
else:
    stamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M%S_%f")
    output = PROJECT_ROOT / "runs/local" / f"dqn_120s_2250m_1000ep_{stamp}"
    output.mkdir(parents=True, exist_ok=False)
    config.save(output / "config.json")
    (output / "run_settings.json").write_text(json.dumps({
        "dqn_settings": asdict(settings), "training_seed": TRAIN_SEED,
        "test_seeds": TEST_SEEDS, "test_weather": TEST_WEATHER,
        "status": "training", "policies": ["dqn", "dyna_dqn"],
        "baseline_run": str(BASE_RUN), "torch_threads": 1,
    }, indent=2))
print("Results:", output)


## Train or load DQN and Dyna-DQN

In [ ]:
agents, histories = {}, {}
for name in ("dqn", "dyna_dqn"):
    marker = output / f"{name}_training_complete.json"
    if marker.exists():
        agent = DQNAgent.load(output / f"{name}.pt")
        if name == "dyna_dqn":
            agent.model.load_state_dict(torch.load(output / "dyna_prediction_model.pt", weights_only=True))
        history = pd.read_csv(output / f"{name}_training.csv")
        print("Loaded", name)
    else:
        agent, history = train_dqn(config, settings, seed=TRAIN_SEED, dyna=(name == "dyna_dqn"))
        agent.save(output / f"{name}.pt")
        history.to_csv(output / f"{name}_training.csv", index=False)
        if name == "dyna_dqn":
            torch.save(agent.model.state_dict(), output / "dyna_prediction_model.pt")
        marker.write_text(json.dumps({"policy": name, "episodes": len(history),
            "training_seed": TRAIN_SEED, "settings": asdict(settings)}, indent=2))
    agents[name], histories[name] = agent, history
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for name, history in histories.items():
    for ax, metric in zip(axes, ["return", "crashed", "energy_j"]):
        ax.plot(history.episode, history[metric].astype(float).rolling(50, min_periods=1).mean(), label=name)
        ax.set(xlabel="Episode", title=metric)
axes[0].legend()
plt.tight_layout()
fig.savefig(output / "training_curves.png", dpi=150)
plt.show()


## Frozen held-out evaluation

Compare crash and completion fractions first, then successful finish time and
sensing power. These are 12 trials from one training seed; they do not establish
statistical significance or overtaking ability. SARSA uses the saved original
adaptive policy. Both neural agents use their existing joint driving/sensing menu.


In [ ]:
agents["adaptive_sarsa"] = LinearExpectedSarsa.load(BASE_RUN / "adaptive_agent.npz")
agents.update({"idle_all_on": None, "random_all_on": None})
evaluation = evaluate_agents(config, agents, TEST_SEEDS, TEST_WEATHER)
summary = evaluation.groupby("policy", sort=False).agg(
    episodes=("seed", "count"), mean_return=("return", "mean"),
    completion_fraction=("completed", "mean"), crash_fraction=("crashed", "mean"),
    mean_completion_time_s=("completion_time_s", "mean"),
    mean_power_w=("mean_power_w", "mean"), mean_distance_m=("distance_m", "mean"))
evaluation.to_csv(output / "evaluation.csv", index=False)
summary.to_csv(output / "summary.csv")
display(summary.round(3))
run_settings = json.loads((output / "run_settings.json").read_text())
run_settings["status"] = "complete"
(output / "run_settings.json").write_text(json.dumps(run_settings, indent=2))
print("Saved:", output)
